In [6]:
!pip install "mlflow==2.22.4" --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 63.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 20.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 88.6 MB/s eta 0:00:0000:0100:01
  Using cached alembic-1.17.2-py3-none-any.whl (248 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.3/14.3 MB 87.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 87.4 MB/s eta 0:00:00:00:0100:01
  Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 7.1 MB/s eta 0:00:0000:0100:01m
  Using cached docker-7.1.0-py3-none-any.whl (147 kB)
  Using cached flask-3.1.2-py3-none-any.whl (103 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 74.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 94.1 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 86.2 MB/s eta 0

In [2]:
import os
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# 1) 设置 MLflow Tracking 地址
#   - 在 docker network 里面，mlflow 服务名就是 “mlflow”，端口 5000
mlflow.set_tracking_uri("http://mlflow:5000")

# 2) 设置 S3 / MinIO 的环境变量（和 docker-compose 里 mlflow 容器保持一致）
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://minio:9000"
os.environ["AWS_ACCESS_KEY_ID"] = "admin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "password"
os.environ["AWS_REGION"] = "us-east-1"

# 3) 选定（或创建）实验
mlflow.set_experiment("demo-from-jupyter")

# 4) 一个最简单的 demo 训练 + 记录
with mlflow.start_run(run_name="rf-demo"):
    db = load_diabetes()
    X_train, X_test, y_train, y_test = train_test_split(
        db.data, db.target, test_size=0.25, random_state=42
    )

    rf = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
    rf.fit(X_train, y_train)

    preds = rf.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = mse ** 0.5

    # log 参数 / metric / 模型
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)
    mlflow.log_metric("rmse", rmse)

    mlflow.sklearn.log_model(rf, artifact_path="model")

print("done, rmse =", rmse)

2025/12/06 13:58:19 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run rf-demo at: http://mlflow:5000/#/experiments/1/runs/d15d940b16c04d6cb82e0cb114844260
🧪 View experiment at: http://mlflow:5000/#/experiments/1
done, rmse = 53.96369145129581


In [4]:
import mlflow
import mlflow.sklearn

# 1. 配置 tracking URI（和你当前一样）
mlflow.set_tracking_uri("http://mlflow:5000")  # 在 spark 容器里用服务名访问

# 2. 指定你刚刚注册的模型
model_name = "rf_diabetes_demo"
model_uri = f"models:/{model_name}/1"   # v1

loaded_model = mlflow.sklearn.load_model(model_uri)
print(loaded_model)

RandomForestRegressor(max_depth=6, random_state=42)


In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("mlflow-iceberg-demo")
    # Iceberg 扩展
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )
    # === 和 Amoro Terminal 日志保持一致的 demo_catalog 配置 ===
    .config("spark.sql.catalog.demo_catalog", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.demo_catalog.catalog-impl", "org.apache.iceberg.rest.RESTCatalog")
    .config("spark.sql.catalog.demo_catalog.uri", "http://iceberg-rest:8181")

    .config("spark.sql.catalog.demo_catalog.table-formats", "ICEBERG")
    .config("spark.sql.catalog.demo_catalog.table.self-optimizing.group", "local")

    .config("spark.sql.catalog.demo_catalog.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.demo_catalog.s3.access-key-id", "admin")
    .config("spark.sql.catalog.demo_catalog.s3.secret-access-key", "password")
    .config("spark.sql.catalog.demo_catalog.client.region", "us-east-1")

    .getOrCreate()
)


25/12/06 15:24:54 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [1]:
spark.sql("SHOW TABLES IN demo_catalog.db").show(truncate=False)

Py4JJavaError: An error occurred while calling o36.sql.
: org.apache.iceberg.exceptions.NoSuchNamespaceException: Namespace does not exist: demo_catalog.db
	at org.apache.iceberg.rest.ErrorHandlers$NamespaceErrorHandler.accept(ErrorHandlers.java:173)
	at org.apache.iceberg.rest.ErrorHandlers$NamespaceErrorHandler.accept(ErrorHandlers.java:166)
	at org.apache.iceberg.rest.HTTPClient.throwFailure(HTTPClient.java:224)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:308)
	at org.apache.iceberg.rest.BaseHTTPClient.get(BaseHTTPClient.java:77)
	at org.apache.iceberg.rest.RESTClient.get(RESTClient.java:97)
	at org.apache.iceberg.rest.RESTSessionCatalog.listTables(RESTSessionCatalog.java:388)
	at org.apache.iceberg.catalog.BaseSessionCatalog$AsCatalog.listTables(BaseSessionCatalog.java:79)
	at org.apache.iceberg.rest.RESTCatalog.listTables(RESTCatalog.java:92)
	at org.apache.iceberg.CachingCatalog.listTables(CachingCatalog.java:136)
	at org.apache.iceberg.spark.SparkCatalog.listT

In [4]:
import os
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_diabetes
from pyspark.sql import functions as F

# 1）MLflow 连接配置（和训练时保持一致）
mlflow.set_tracking_uri("http://mlflow:5000")

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://minio:9000"
os.environ["AWS_ACCESS_KEY_ID"] = "admin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "password"
os.environ["AWS_REGION"] = "us-east-1"

# 2）开始一个“预测”用的 run，方便记录 run_id
with mlflow.start_run(run_name="rf-predict-demo"):

    # 2.1 从注册中心加载已经训练好的模型
    #    假设你在 UI 里注册的名字叫 rf_diabetes_demo，版本是 1
    model = mlflow.sklearn.load_model("models:/rf_diabetes_demo/1")

    # 2.2 特征直接用 diabetes 自带的数据，写死在代码里，不走 Iceberg
    db = load_diabetes()
    # 这里随便取前 50 条做一个 demo
    X = db.data[:50]

    # 2.3 构造一个“用户 id”，假装这些样本对应的用户
    user_ids = [f"user_{i}" for i in range(len(X))]

    # 2.4 调用模型做预测（回归问题，这里就是预测一个数值）
    preds = model.predict(X)

    # 2.5 取当前这个预测 run 的 run_id，用来写回表里做追溯
    run_id = mlflow.active_run().info.run_id

    # 3）把预测结果变成 Spark DataFrame
    #    注意这里只关心回写需要的几个字段：user_id / score / model_xxx / run_id
    rows = [
        (user_ids[i], float(preds[i]), "rf_diabetes_demo", "1", run_id)
        for i in range(len(preds))
    ]

    pred_sdf = (
        spark.createDataFrame(
            rows,
            ["user_id", "score", "model_name", "model_version", "run_id"]
        )
        .withColumn("predict_time", F.current_timestamp())
        .withColumn("label_pred", F.lit(None).cast("int"))  # 先写 null
    )

    # 4）写回到 Iceberg 表 demo_catalog.db.tb_user_churn_pred
    #    分区已经按 predict_time(days) 配好了，这里只需要正常 append 即可
    (
        pred_sdf
        .select(
            "user_id",
            "predict_time",
            "score",
            "label_pred",      # ⭐ 现在包含这列了
            "model_name",
            "model_version",
            "run_id",
        )
        .writeTo("demo_catalog.db.tb_user_churn_pred")
        .append()
    )

    print("写回完成，记录数 =", pred_sdf.count())
    print("本次预测 run_id =", run_id)


写回完成，记录数 = 50
本次预测 run_id = 0d075f549b0343d197d4c2adfd6a82f5
🏃 View run rf-predict-demo at: http://mlflow:5000/#/experiments/0/runs/0d075f549b0343d197d4c2adfd6a82f5
🧪 View experiment at: http://mlflow:5000/#/experiments/0
